#### All matches in the admin data
#### coverage of matches
#### selection of relevant matches
#### addition from unmatched but relevant gmaps pids
#### final pool list

In [ ]:
import pandas as pd

In [ ]:
relevant_place_types = [
    'aquatic center','athletic club','amusement park','beach',
    'indoor swimming pool','lake','lake shore swimming area',
    'lido','outdoor bath','outdoor swimming pool','public bath','public bathroom',
    'public wheelchair-accessible bathroom','public sauna','public beach','public swimming pool',
    'recreation center','sauna','sauna club','spa','spa and health club','sports complex',
    'swim club','swimming basin','swimming facility','swimming lake','swimming pool','swimming school',
    'thermal baths','water','water park']

### all matches and coverage

In [ ]:
### original pools from the administrative  website
admin_pools_cleaned = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/groundtruth_pools/baederleben_pools_16112025_cleaned.csv')

admin_pools_cleaned.info()  

In [ ]:
admin_pools_cleaned.pool_id.nunique()  #3884 unique pools

In [ ]:
#distribution of pool types
admin_pools_cleaned['pool_type'].value_counts()

In [ ]:
#all  matched admin pools in 500m radius and on address
matched_500m = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates_24112025/closest_matches_only_all_columns.csv')
matched_address = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates_24112025/complete_ADDRESS_matches_all_columns.csv')
### all matches and coverage
all_matches = pd.concat([matched_500m, matched_address]).drop_duplicates(subset=['pool_id', 'pool_name']).reset_index(drop=True)
all_matches.info()

In [ ]:
all_matches.pool_id.nunique()

In [ ]:
all_matches.pool_type.value_counts()

##### Total of 24 admin pools do not match  to any gmaps point in 500m radius or on addresses
##### 5 Hallenbads, 8 Naturbads and 11 freibads

### relevant matches and coverage

##### While almost all the admin pools have matched to some gmaps pid withing 500m or on address, the corresponding gmaps point may not be relevant  (eg. bus stop). Now  we selec those for which the  match is relevant for our work

In [ ]:
relevant_matches = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/data/admin_matced_relevant_25112025.csv')
relevant_matches = relevant_matches.drop_duplicates(subset=['pool_id']).reset_index(drop=True)
relevant_matches = relevant_matches.drop_duplicates(subset=['pool_name','place_id']).reset_index(drop=True)
relevant_matches.info()

In [ ]:
relevant_matches.pool_id.nunique()

In [ ]:
not_in_relevant = all_matches[~all_matches['pool_id'].isin(relevant_matches['pool_id'])]
not_in_relevant['pool_type'].value_counts()

In [ ]:
not_relevant_but_rev = not_in_relevant[
    (not_in_relevant['primary_place_type'].str.lower().isin(relevant_place_types)) |
    (not_in_relevant['place_subtype'].apply(
        lambda x: any(item.lower() in relevant_place_types for item in x) 
        if isinstance(x, list) else False
    ))
].reset_index(drop=True)
not_relevant_but_rev.info()

In [ ]:
not_relevant_but_rev.pool_type.value_counts()

In [ ]:
#add to relevant matches
relevant_matches_updated = pd.concat([relevant_matches, not_relevant_but_rev]).drop_duplicates(subset=['pool_id']).reset_index(drop=True)
relevant_matches_updated.info()

In [ ]:
relevant_matches_updated.pool_id.nunique()

#### checking for pools not in relevant but in the multiple matches list

In [ ]:
multiple_matches= pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates_24112025/complete_ALL_matches_all_columns.csv')
multiple_matches = pd.concat([multiple_matches, matched_address])
multiple_matches.info()

In [ ]:
multiple_matches.pool_id.nunique()

In [ ]:
multiple_matches_not_in_relevant = multiple_matches[~multiple_matches['pool_id'].isin(relevant_matches_updated['pool_id'])]
multiple_matches_not_in_relevant['pool_type'].value_counts()

In [ ]:
rev_multiple_matches_not_in_relevant = multiple_matches_not_in_relevant[
    (multiple_matches_not_in_relevant['primary_place_type'].str.lower().isin(relevant_place_types)) |
    (multiple_matches_not_in_relevant['place_subtype'].apply(
        lambda x: any(item.lower() in relevant_place_types for item in x) 
        if isinstance(x, list) else False
    ))
].reset_index(drop=True)
rev_multiple_matches_not_in_relevant.info()

In [ ]:
##update relevant matches again
relevant_matches_final = pd.concat([relevant_matches_updated, rev_multiple_matches_not_in_relevant]).drop_duplicates(subset=['pool_id']).reset_index(drop=True)
relevant_matches_final = relevant_matches_final.drop_duplicates(subset=['pool_id']).reset_index(drop=True)
relevant_matches_final.info()

In [ ]:
relevant_matches_final.pool_id.nunique()

In [ ]:
relevant_matches_final.pool_type.value_counts()

In [ ]:
probably_irrelevant = all_matches[~all_matches['pool_id'].isin(relevant_matches_final['pool_id'])]
probably_irrelevant.info()

In [ ]:
probably_irrelevant.pool_id.nunique()

In [ ]:
3886 + 1236

In [ ]:
all_matches_irrelevant = multiple_matches[~multiple_matches['pool_id'].isin(relevant_matches_final['pool_id'])]
all_matches_irrelevant.info()

In [ ]:
all_matches_irrelevant.pool_id.nunique()

In [ ]:
all_matches_irrelevant.drop_duplicates(subset=['pool_id']).pool_type.value_counts()

### Now lets check for relevant  places in the unmatched google and see if there are matches on address to the admin pools with irrelevant matches

In [ ]:
unmatched_google = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates_24112025/unmatched_gmaps_points_all_columns.csv')
unmatched_google = unmatched_google.drop_duplicates(subset=['place_id']).reset_index(drop=True)
unmatched_google.info()

### matching 1236 admin pools on address in the unmatched google map places again

In [ ]:
from tqdm import tqdm
import pandas as pd
from fuzzywuzzy import fuzz

def fuzzy_match_dataframes_fast(df_a, df_b, address_col='address', 
                              threshold=80, scorer=fuzz.token_set_ratio,
                              include_all_a=True, show_progress=True):
    """
    Faster version with progress tracking.
    """
    print(f"Matching {len(df_a)} rows against {len(df_b)} rows...")
    print(f"Total comparisons: {len(df_a) * len(df_b):,}")
    
    # Create copies
    df_a = df_a.copy().reset_index(drop=True)
    df_b = df_b.copy().reset_index(drop=True)
    
    matches = []
    
    # Use tqdm for progress bar
    iterator = range(len(df_a))
    if show_progress:
        iterator = tqdm(iterator, desc="Matching addresses")
    
    for idx_a in iterator:
        row_a = df_a.iloc[idx_a]
        address_a = str(row_a[address_col])
        best_matches = []
        
        for idx_b in range(len(df_b)):
            address_b = str(df_b.iloc[idx_b][address_col])
            score = scorer(address_a, address_b)
            
            if score >= threshold:
                best_matches.append((idx_b, score))
        
        if best_matches:
            for idx_b, score in best_matches:
                row_b = df_b.iloc[idx_b]
                combined_row = {}
                
                # Add A columns
                for col in df_a.columns:
                    combined_row[f'{col}_a'] = row_a[col]
                
                # Add B columns  
                for col in df_b.columns:
                    combined_row[f'{col}_b'] = row_b[col]
                
                combined_row['match_score'] = score
                combined_row['match_type'] = 'fuzzy_match'
                matches.append(combined_row)
                
        elif include_all_a:
            # No matches
            combined_row = {}
            for col in df_a.columns:
                combined_row[f'{col}_a'] = row_a[col]
            for col in df_b.columns:
                combined_row[f'{col}_b'] = None
            combined_row['match_score'] = 0
            combined_row['match_type'] = 'no_match'
            matches.append(combined_row)
    
    result_df = pd.DataFrame(matches)
    print(f"Matching complete. Found {len(result_df)} total rows ({len([m for m in matches if m['match_score'] > 0])} matches)")
    return result_df



In [ ]:
# Rename and process in one go
def process_column_name(df ,col):
    col = col.replace('_a', '_admin').replace('_b', '_gmaps')
    
    if col.endswith('_admin'):
        base = col[:-6]
        # Check if gmaps version exists
        gmaps_col = base + '_gmaps'
        if gmaps_col in [c.replace('_a', '_admin').replace('_b', '_gmaps') for c in df.columns]:
            return col
        else:
            return base
    elif col.endswith('_gmaps'):
        base = col[:-6]
        # Check if admin version exists
        admin_col = base + '_admin'
        if admin_col in [c.replace('_a', '_admin').replace('_b', '_gmaps') for c in df.columns]:
            return col
        else:
            return base
    else:
        return col

In [ ]:
addmin_to_match = admin_pools_cleaned[admin_pools_cleaned['pool_id'].isin(probably_irrelevant['pool_id'])]
addmin_to_match.info()

In [ ]:
# Test with your data
df_a = addmin_to_match
df_b = unmatched_google

print(f"DataFrame A: {len(df_a)} rows")
print(f"DataFrame B: {len(df_b)} rows")

# If datasets are large, try these optimizations:
if len(df_a) * len(df_b) > 1000000:  # If over 1M comparisons
    print("Large dataset detected - consider:")
    print("1. Using a higher threshold to reduce matches")
    print("2. Sampling smaller subsets for testing")
    print("3. Using exact matching first to reduce fuzzy matching load")

# Run with progress bar
result = fuzzy_match_dataframes_fast(df_a, df_b, threshold=80)

In [ ]:
#the  unmatched 24
admin_24 = admin_pools_cleaned[~admin_pools_cleaned['pool_id'].isin(all_matches['pool_id'])]
admin_24.info()

In [ ]:
result_24.info()

In [ ]:
result_24.pool_id_a.nunique()

In [ ]:
# remove column suffices _a and _b
result.columns = [col.replace('_a', '').replace('_b', '') for col in result.columns]

#change the  first address column name 
result_cols = result.columns.tolist()
new_names = result_cols
new_names[1] = 'city_admin'
new_names[6] = 'latitude_admin'
new_names[7] = 'longitude_admin'
new_names[9] = 'address_admin'
new_names[13] = 'address_gmaps'
new_names[20] = 'latitude_gmaps'
new_names[21] = 'longitude_gmaps'
new_names[43] = 'city_gmaps'
result.columns = new_names

result.info()

In [ ]:
result.pool_id.nunique()

In [ ]:
result.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/data/admin_1236_on_gmaps_unmatched_addresses_25112025.csv', index=False)

In [ ]:
###select relevant places from unmatched google places based on place types
relevant_unmatched_google = unmatched_google[
    (unmatched_google['primary_place_type'].str.lower().isin(relevant_place_types)) |
    (unmatched_google['place_subtype'].apply(
        lambda x: any(item.lower() in relevant_place_types for item in x) 
        if isinstance(x, list) else False
    ))
].reset_index(drop=True)
relevant_unmatched_google.info()

In [ ]:
## 161 poolsin probably irrelevant matched to relevant unmatched google places
result_relevant = result[result['place_id'].isin(relevant_unmatched_google['place_id'])]
result_relevant.info()

In [ ]:
result_relevant[['pool_id','pool_name','address','place_id','place_name','address','primary_place_type','place_subtype','match_score','match_type']]

In [ ]:
result_relevant.pool_id.nunique()

In [ ]:
relevant_matches_final.columns

In [ ]:
rev_cols =['pool_id', 'city_admin', 'pool_type',
       'pool_name', 'street', 'postal_code', 'latitude_admin',
       'longitude_admin', 'status', 'address_admin', 'place_id',
       'unique_identifier', 'place_name', 'owner', 'borough', 'place_rating',
       'total_reviews', 'primary_place_type', 'place_subtype', 'website',
       'google_url', 'popular_price_range', 'place_phone', 'higlights',
       'popular_for', 'accessibility', 'offerings', 'dining_options',
       'amenities', 'atmosphere', 'crowd_level', 'planning', 'payment_options',
       'children', 'parking', 'index_right', 'WKR_NR', 'WKR_NAME', 'LAND_NR',
       'LAND_NAME', 'cluster_id', 'public', 'other_info', 'retrieved_time','opening_hours', 'price_ranges', 'service_options', 'geometry',
       'address_gmaps', 'latitude_gmaps',
       'longitude_gmaps', 'city_gmaps',]

In [ ]:
len(rev_cols)

In [ ]:
result_relevant.columns

In [ ]:
len(result_relevant.columns)

In [ ]:
#elements in rev_cols not in result_relevant.columns
set(rev_cols) - set(result_relevant.columns)    

In [ ]:
all_possible_relevant_matches = pd.concat([relevant_matches_final[rev_cols], result_relevant]).drop_duplicates(subset=['pool_id']).reset_index(drop=True)
all_possible_relevant_matches.info()

In [ ]:
all_possible_relevant_matches.pool_id.nunique()

In [ ]:
###now all possibly irrelevant pools updated
all_matches_irrelevant_updated = all_matches_irrelevant[~all_matches_irrelevant['pool_id'].isin(all_possible_relevant_matches['pool_id'])]
all_matches_irrelevant_updated.info()

In [ ]:
all_matches_irrelevant_updated.pool_id.nunique()

#### update unmatched google places

In [ ]:
unmatched_google_updated = unmatched_google[~unmatched_google['place_id'].isin(all_possible_relevant_matches['place_id'])]
unmatched_google_updated.info()

### What if we do adress matching for the remaing 1236 + the 24 on all the google maps data points?

In [ ]:
### all google pools collected and merged
import geopandas as gpd
gmaps_pools = gpd.read_file('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/clean_datasets/merged_all_gmaps_pids_24112025.gpkg')
gmaps_pools.info()

In [ ]:
# Test with your data
df_a = addmin_to_match
df_c = gmaps_pools

print(f"DataFrame A: {len(df_a)} rows")
print(f"DataFrame B: {len(df_c)} rows")

# If datasets are large, try these optimizations:
if len(df_a) * len(df_c) > 1000000:  # If over 1M comparisons
    print("Large dataset detected - consider:")
    print("1. Using a higher threshold to reduce matches")
    print("2. Sampling smaller subsets for testing")
    print("3. Using exact matching first to reduce fuzzy matching load")

# Run with progress bar
result_gmaps_all = fuzzy_match_dataframes_fast(df_a, df_c, threshold=80)

In [ ]:
admin_24= admin_pools_cleaned[~admin_pools_cleaned['pool_id'].isin(relevant_matches_final['pool_id'])]

In [ ]:
df_d = admin_24 
df_c = unmatched_google 

print(f"DataFrame A: {len(df_d)} rows")
print(f"DataFrame B: {len(df_c)} rows")

# If datasets are large, try these optimizations:
if len(df_d) * len(df_c) > 1000000:  # If over 1M comparisons
    print("Large dataset detected - consider:")
    print("1. Using a higher threshold to reduce matches")
    print("2. Sampling smaller subsets for testing")
    print("3. Using exact matching first to reduce fuzzy matching load")

# Run with progress bar
result_24 = fuzzy_match_dataframes_fast(df_d, df_c, threshold=80)

In [ ]:
result_gmaps_all.columns = [process_column_name(result_gmaps_all,col) for col in result_gmaps_all.columns]
result_24.columns = [process_column_name(result_24,col) for col in result_24.columns]

In [ ]:
#merge both results
final_result_1260 = pd.concat([result_gmaps_all, result_24]).reset_index(drop=True)
final_result_1260.info()

In [ ]:
final_result_1260.place_id.nunique()

In [ ]:
#selecting relevant matches for the 1236
rev_final_result_1260 = final_result_1260[
    (final_result_1260['primary_place_type'].str.lower().isin(relevant_place_types)) |
    (final_result_1260['place_subtype'].apply(
        lambda x: any(item.lower() in relevant_place_types for item in x) 
        if isinstance(x, list) else False
    ))
].reset_index(drop=True)
rev_final_result_1260.info()

In [ ]:
rev_final_result_1260.place_id.nunique()

In [ ]:
google_unmatched_rev = unmatched_google[
    (unmatched_google['primary_place_type'].str.lower().isin(relevant_place_types)) |
    (unmatched_google['place_subtype'].apply(
        lambda x: any(item.lower() in relevant_place_types for item in x) 
        if isinstance(x, list) else False
    ))
].reset_index(drop=True)
google_unmatched_rev.info()

In [ ]:
#updated multiple matches 

multiple_matches_updated = pd.concat([multiple_matches, final_result_1260])
multiple_matches_updated.info()

In [ ]:
multiple_matches_updated.pool_id.nunique()

### all the found matches and the ones that can

In [ ]:
every_possible_relevant_matches = pd.concat([all_possible_relevant_matches[rev_cols], rev_final_result_1260]).drop_duplicates(subset=['pool_id']).reset_index(drop=True)
every_possible_relevant_matches.info()

In [ ]:
every_possible_relevant_matches.pool_id.nunique()

In [ ]:
every_possible_relevant_matches.pool_type.value_counts()

In [ ]:
all_matches_irrelevant_updated = pd.concat([all_matches_irrelevant,final_result_1260])
all_matches_irrelevant_updated = all_matches_irrelevant_updated[~all_matches_irrelevant_updated['pool_id'].isin(every_possible_relevant_matches['pool_id'])]
all_matches_irrelevant_updated = all_matches_irrelevant_updated.dropna(subset=['place_id'])
all_matches_irrelevant_updated.info()

In [ ]:
all_matches_irrelevant_updated.pool_id.nunique()

In [ ]:
every_possible_relevant_matches = every_possible_relevant_matches.dropna(subset=['place_id'])

In [ ]:
every_possible_relevant_matches.pool_id.nunique()

In [ ]:
unmatched_admin_pools = admin_pools_cleaned[~admin_pools_cleaned['pool_id'].isin(every_possible_relevant_matches['pool_id'])]
unmatched_admin_pools = unmatched_admin_pools[~unmatched_admin_pools['pool_id'].isin(all_matches_irrelevant_updated['pool_id'])]
unmatched_admin_pools.info() 

In [ ]:
unmatched_admin_pools.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/data/admin_unmatched_final_19_25112025.csv', index=False)

In [ ]:
all_matches_irrelevant_updated.drop_duplicates(subset=['pool_id']).pool_type.value_counts()

In [ ]:
unmatched_google_updated = unmatched_google[~unmatched_google['place_id'].isin(every_possible_relevant_matches['place_id'])]
unmatched_google_updated = unmatched_google_updated[~unmatched_google_updated['place_id'].isin(['place_id'])]
unmatched_google_updated = unmatched_google_updated.drop_duplicates(subset=['place_id']).reset_index(drop=True)
unmatched_google_updated.info()

In [ ]:
google_unmatched_rev = unmatched_google_updated[
    (unmatched_google_updated['primary_place_type'].str.lower().isin(relevant_place_types)) |
    (unmatched_google_updated['place_subtype'].apply(
        lambda x: any(item.lower() in relevant_place_types for item in x) 
        if isinstance(x, list) else False
    ))
].reset_index(drop=True)
google_unmatched_rev.info()

In [ ]:
google_unmatched_rev = google_unmatched_rev[~google_unmatched_rev['place_id'].isin(every_possible_relevant_matches['place_id'])].reset_index(drop=True)
google_unmatched_rev.info()

In [ ]:
unmatched_google_updated = unmatched_google[~unmatched_google['place_id'].isin(google_unmatched_rev['place_id'])]
unmatched_google_updated = unmatched_google_updated[~unmatched_google_updated['place_id'].isin(every_possible_relevant_matches['place_id'])]
unmatched_google_updated = unmatched_google_updated.drop_duplicates(subset=['place_id']).reset_index(drop=True)
unmatched_google_updated.info()

In [ ]:
google_unmatched_rev.place_id.nunique()

In [ ]:
google_unmatched_rev[google_unmatched_rev['place_id'].isin(every_possible_relevant_matches['place_id'])]

In [ ]:
unmatched_google_updated.place_id.nunique()

In [ ]:
##cross checking
gmaps_not_mapped = gmaps_pools[~gmaps_pools['place_id'].isin(every_possible_relevant_matches['place_id'])]
gmaps_not_mapped = gmaps_not_mapped[~gmaps_not_mapped['place_id'].isin(all_matches_irrelevant_updated['place_id'])]
gmaps_not_mapped =  gmaps_not_mapped[~gmaps_not_mapped['place_id'].isin(google_unmatched_rev['place_id'])]
gmaps_not_mapped = gmaps_not_mapped.drop_duplicates(subset=['place_id']).reset_index(drop=True)
gmaps_not_mapped.info()

In [ ]:
gmaps_pools[~gmaps_pools['place_id'].isin(gmaps_not_mapped['place_id'])].info()

In [ ]:
gmaps_not_mapped.place_id.nunique()

### save relevant matches and probably irrelevant matches and crosscheck again manually

In [ ]:
#4271 unique pool ids
every_possible_relevant_matches.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/data/admin_matched_relevant_final_4276_25112025.csv', index=False)


In [ ]:
#851 unique pool ids
all_matches_irrelevant_updated.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/data/admin_matched_irrelevant_final_851_25112025.csv', index=False)

In [ ]:
#rev_ google  unmatches 943
google_unmatched_rev.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/data/google_unmatched_relevant_final_943_25112025.csv', index=False)

In [ ]:
#unmatched google probably irrelevant 2213 places
gmaps_not_mapped.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/data/google_unmatched_irrelevant_final_4903_25112025.csv', index=False)